## **Malay-English Hallucination Detection**

This notebook implements a supervised NLP classification model to detect truthful and hallucinated answers using a Malay-English HaluEval dataset.

###### 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

from scipy.sparse import hstack, csr_matrix

###### 2. Upload and Load Dataset

In [ ]:
uploaded = files.upload()

df = pd.read_csv("englishmalay_halueval.csv")

df.head()

###### 3. Dataset Checking

In [ ]:
print("Dataset shape:", df.shape)

print("\nLanguage distribution:")
print(df["language"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

###### 4. Data Cleaning

In [ ]:
df = df.dropna(subset=["question", "answer", "label", "context"])
df = df.drop_duplicates(subset=["question", "answer", "label", "language"])

print("Dataset shape after cleaning:", df.shape)

###### 5. Combine Context, Question and Answer

In [ ]:
df["text"] = (
    "Context: " + df["context"].astype(str) +
    " Question: " + df["question"].astype(str) +
    " Answer: " + df["answer"].astype(str)
)

df[["text", "label"]].head()

###### 6. Create Context-Based Features

In [ ]:
def get_words(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\u00C0-\u024F\u1E00-\u1EFF0-9\s]", " ", text)
    words = text.split()
    return set(words)

df["answer_length"] = df["answer"].astype(str).apply(lambda x: len(x.split()))
df["question_length"] = df["question"].astype(str).apply(lambda x: len(x.split()))
df["context_length"] = df["context"].astype(str).apply(lambda x: len(x.split()))

def overlap_ratio(answer, context):
    answer_words = get_words(answer)
    context_words = get_words(context)

    if len(answer_words) == 0:
        return 0

    overlap = answer_words.intersection(context_words)
    return len(overlap) / len(answer_words)

df["answer_context_overlap"] = df.apply(
    lambda row: overlap_ratio(row["answer"], row["context"]),
    axis=1
)

df["answer_in_context"] = df.apply(
    lambda row: 1 if str(row["answer"]).lower() in str(row["context"]).lower() else 0,
    axis=1
)

df[[
    "answer_length",
    "question_length",
    "context_length",
    "answer_context_overlap",
    "answer_in_context"
]].head()

###### 7. Define Input Features and Target Label

In [ ]:
X_text = df["text"]

X_numeric = df[[
    "answer_length",
    "question_length",
    "context_length",
    "answer_context_overlap",
    "answer_in_context"
]]

y = df["label"]

###### 8 . Train/test split

In [ ]:
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text,
    X_numeric,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_text_train))
print("Testing samples:", len(X_text_test))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

###### 9. TF-IDF Feature Representation

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    lowercase=True,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_text_train)
X_test_tfidf = vectorizer.transform(X_text_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

###### 10. Combine TF-IDF and Numeric Features

In [ ]:
X_train_num = csr_matrix(X_num_train.values)
X_test_num = csr_matrix(X_num_test.values)

X_train_final = hstack([X_train_tfidf, X_train_num])
X_test_final = hstack([X_test_tfidf, X_test_num])

print("Final training feature shape:", X_train_final.shape)
print("Final testing feature shape:", X_test_final.shape)

###### 11. Train Logistic Regression Model

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_final, y_train)

print("Model training completed.")

###### 12. Make Predictions

In [ ]:
y_pred = model.predict(X_test_final)

print("Prediction completed.")

###### 13. Evaluate Model Performance

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Evaluation Results")
print("------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Truthful", "Hallucinated"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))